# Complete Pipeline: Evidential Classifier Chains (EDL-ECC) & Multi-Dataset Benchmark

Notebook này được thiết kế theo 2 chế độ:
- **Chế độ 1 (Deep-Dive Single Dataset)**: Chạy từng bước chi tiết (EDA $\rightarrow$ Preprocessing $\rightarrow$ EDL-ECC $\rightarrow$ Uncertainty $\rightarrow$ Visualizations) cho 1 tập dữ liệu được chọn.
- **Chế độ 2 (Multi-Dataset Benchmark)**: Tự động chạy toàn bộ 9 tập dữ liệu trong thư mục `data/` ở Cell cuối cùng!

Các bước triển khai:
1. **EDA**: Hỗ trợ đọc cả ARFF Dạng Dày (Dense) & Dạng Thưa (Sparse), phân tích tương quan nhãn và mẫu dữ liệu.
2. **Preprocessing**: Standardization đặc trưng, chia tập Train/Val (80/20), PyTorch DataLoader.
3. **EDL Model Architecture & Loss Function**: Bằng chứng Dirichlet $\boldsymbol{\alpha} = e + 1.0$, xác suất $p$, độ bất định $u$, và EDL Expected MSE Loss + KL Divergence.
4. **Ensemble Classifier Chains (EDL-ECC)**: Kiến trúc chuỗi phân loại EDL với Uncertainty-Gating.
5. **Training & Uncertainty Calibration**: Huấn luyện thực thụ trên PyTorch, phân tích độ bất định giữa dự đoán đúng và sai.
6. **Baseline Comparison**: So sánh thực tế với Binary Relevance (BR), Classifier Chains (CC), Standard RAkEL, EDL-ECC, và EDL-RAkEL.
7. **Multi-Dataset Loop**: Tự động thực thi toàn bộ 9 tập dữ liệu (`Scene`, `Yeast`, `emotions`, `HumanPseAAC`, `PlantPseAAC`, `GpositivePseAAC`, `VirusPseAAC`, `Water-quality`, `CHD_49`).


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.io import arff
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, hamming_loss, jaccard_score, precision_score, recall_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
os.makedirs('./outputs', exist_ok=True)

# 9 Benchmark Datasets Config
DATASET_CONFIGS = {
    'Scene': {'file': 'Scene.arff', 'num_labels': 6},
    'Yeast': {'file': 'Yeast.arff', 'num_labels': 14},
    'emotions': {'file': 'emotions.arff', 'num_labels': 6},
    'HumanPseAAC': {'file': 'HumanPseAAC.arff', 'num_labels': 14},
    'PlantPseAAC': {'file': 'PlantPseAAC.arff', 'num_labels': 12},
    'GpositivePseAAC': {'file': 'GpositivePseAAC.arff', 'num_labels': 4},
    'VirusPseAAC': {'file': 'VirusPseAAC.arff', 'num_labels': 6},
    'Water-quality': {'file': 'Water-quality.arff', 'num_labels': 14},
    'CHD_49': {'file': 'CHD_49.arff', 'num_labels': 6}
}

# Select primary dataset for deep-dive step-by-step walkthrough:
DATASET_NAME = 'Scene'
num_labels = DATASET_CONFIGS[DATASET_NAME]['num_labels']
dataset_path = Path(f"./data/{DATASET_CONFIGS[DATASET_NAME]['file']}")

print(f"✓ Selected Primary Dataset: {DATASET_NAME} ({dataset_path}, Num Labels: {num_labels})")
print(f"✓ Available Datasets for Batch Benchmark: {list(DATASET_CONFIGS.keys())}")


## BƯỚC 1: Khám phá & Trực quan hóa dữ liệu (EDA)

### 1.1 Robust ARFF Loader (Tự động hỗ trợ cả Dense và Sparse ARFF)


In [ ]:
def load_arff_robust(path, num_labels):
    path = Path(path)
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    
    is_sparse = any(l.strip().startswith('{') for l in lines[:100] if l.strip())
    
    if is_sparse:
        attributes = []
        for line in lines:
            if line.strip().lower().startswith('@attribute'):
                parts = line.split()
                if len(parts) >= 2: attributes.append(parts[1])
        
        start_data = next(i for i, line in enumerate(lines) if line.strip().lower() == '@data')
        rows = []
        for line in lines[start_data + 1:]:
            line = line.strip()
            if not line or line.startswith('%'): continue
            if line.startswith('{') and line.endswith('}'):
                row = {}
                for item in line[1:-1].split(','):
                    item = item.strip()
                    if not item: continue
                    parts = item.split()
                    if len(parts) >= 2:
                        idx, val = int(parts[0]), float(parts[1])
                        row[idx] = val
                rows.append(row)
            else: rows.append({})
        
        df = pd.DataFrame(0.0, index=range(len(rows)), columns=range(len(attributes)))
        for i, row in enumerate(rows):
            for idx, val in row.items():
                if 0 <= idx < len(attributes): df.iat[i, idx] = val
        
        X = df.iloc[:, :-num_labels].values.astype('float32')
        Y = df.iloc[:, -num_labels:].values.astype('float32')
    else:
        data, meta = arff.loadarff(path)
        df = pd.DataFrame(data)
        for col in df.columns:
            if df[col].dtype == object:
                try: df[col] = df[col].str.decode('utf-8')
                except: pass
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
        X = df.iloc[:, :-num_labels].values.astype('float32')
        Y = df.iloc[:, -num_labels:].values.astype('float32')
        Y = (Y > 0).astype('float32')
        
    return X, Y

X_full, Y_full = load_arff_robust(dataset_path, num_labels)

print(f'✓ Primary Dataset loaded: {DATASET_NAME}')
print(f'✓ Features shape: {X_full.shape}')
print(f'✓ Labels shape: {Y_full.shape}')
print(f'✓ Label sparsity: {Y_full.mean()*100:.2f}%')


### 1.2 Label Distribution & Label Correlation Heatmap

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Label counts
label_counts = Y_full.sum(axis=0)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(label_counts)))
ax1.bar(range(len(label_counts)), label_counts, color=colors, edgecolor='black')
ax1.set_xticks(range(len(label_counts)))
ax1.set_xticklabels([f'L{i+1}' for i in range(len(label_counts))])
ax1.set_ylabel('Số lượng mẫu mang nhãn', fontsize=11)
ax1.set_title(f'Phân bố Nhãn tập dữ liệu {DATASET_NAME}', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Correlation heatmap
corr_matrix = np.corrcoef(Y_full.T)
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=True if len(label_counts)<=10 else False, fmt='.2f',
            xticklabels=[f'L{i+1}' for i in range(len(label_counts))],
            yticklabels=[f'L{i+1}' for i in range(len(label_counts))],
            ax=ax2, vmin=-1, vmax=1)
ax2.set_title('Ma trận Tương quan giữa các Nhãn (Chứng minh cần CC/RAkEL)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'./outputs/1_eda_{DATASET_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()


## BƯỚC 2: Tiền xử lý & PyTorch DataLoader

In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_full)

X_train, X_val, Y_train, Y_val = train_test_split(
    X_norm, Y_full, test_size=0.2, random_state=42
)

# Safety check for single-class labels
if any(len(np.unique(Y_train[:, col])) < 2 for col in range(Y_train.shape[1])):
    dummy_X = np.zeros((2, X_train.shape[1]), dtype='float32')
    dummy_Y = np.zeros((2, Y_train.shape[1]), dtype='float32')
    dummy_Y[1, :] = 1.0
    X_train = np.vstack([X_train, dummy_X])
    Y_train = np.vstack([Y_train, dummy_Y])

batch_size = 64
train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(Y_train).float())
val_ds = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(Y_val).float())

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

print(f'✓ Train set: {X_train.shape}, Val set: {X_val.shape}')


## BƯỚC 3: Kiến trúc Mô hình EDL (Evidential Deep Learning) & Loss Function

Mô hình EDL cho phân loại nhị phân đa nhãn xuất ra tham số Dirichlet $\boldsymbol{\alpha} = (\alpha_0, \alpha_1)$ cho mỗi nhãn:
- Bằng chứng (Evidence): $e = \text{ReLU}(\text{out}) + 1\text{e-}4$
- Tham số Dirichlet: $\alpha = e + 1.0$
- Xác suất nhãn dương: $p = \frac{\alpha_1}{\alpha_0 + \alpha_1}$
- Độ bất định (Uncertainty): $u = \frac{2}{\alpha_0 + \alpha_1} \in [0, 1]$


In [ ]:
class EDLModel(nn.Module):
    """Evidential Deep Learning model for multi-label binary classification."""
    def __init__(self, in_dim, num_labels, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.out = nn.Linear(hidden // 2, num_labels * 2)

    def forward(self, x):
        h = self.net(x)
        e = F.relu(self.out(h)) + 1e-4
        e = e.view(x.size(0), -1, 2)
        alpha = e + 1.0
        return alpha

def dirichlet_kl_binary(alpha):
    """KL divergence between Dir(alpha) and uniform Dir(1)."""
    beta = torch.ones_like(alpha)
    S_alpha = torch.sum(alpha, dim=2, keepdim=True)
    S_beta = torch.sum(beta, dim=2, keepdim=True)
    
    lnB_alpha = torch.sum(torch.lgamma(alpha), dim=2, keepdim=True) - torch.lgamma(S_alpha)
    lnB_beta = torch.sum(torch.lgamma(beta), dim=2, keepdim=True) - torch.lgamma(S_beta)
    digamma_diff = torch.digamma(alpha) - torch.digamma(S_alpha)
    
    kl = torch.sum((alpha - beta) * digamma_diff, dim=2, keepdim=True) + lnB_alpha - lnB_beta
    return kl.squeeze(2)

def edl_mse_loss(alpha, target, epoch, annealing_step=5):
    """Expected MSE + KL Divergence Loss for EDL."""
    S = torch.sum(alpha, dim=2, keepdim=True)
    p = alpha / S
    y = torch.stack([1.0 - target.float(), target.float()], dim=2)
    
    mse = torch.sum((y - p) ** 2, dim=2)
    var_term = torch.sum(p * (1.0 - p) / (S + 1.0), dim=2)
    expected_err = mse + var_term
    
    weight = torch.where(target > 0, 3.0, 1.0)
    expected_err = expected_err * weight
    
    kl = dirichlet_kl_binary(alpha)
    lambda_t = min(1.0, epoch / max(1, annealing_step))
    
    loss = expected_err + lambda_t * kl
    return loss.mean()

def predict_proba_and_uncertainty(alpha):
    S = alpha.sum(dim=2)
    p_pos = alpha[:, :, 1] / S
    u = 2.0 / S
    return p_pos, u

print("✓ EDL Model Architecture & Loss Functions defined successfully!")


## BƯỚC 4: Kiến trúc Ensemble Classifier Chains (EDL-ECC) với Evidential Uncertainty-Gating

In [ ]:
class EDL_ECC:
    """Ensemble Classifier Chains with EDL Base Learners & Uncertainty-Gating."""
    def __init__(self, in_dim, num_labels, n_chains=3, hidden=256, device='cpu'):
        self.in_dim = in_dim
        self.num_labels = num_labels
        self.n_chains = n_chains
        self.hidden = hidden
        self.device = device
        self.chains = []
        self.orders = []

    def fit(self, X_tr, Y_tr, epochs=10, lr=1e-3):
        X_tr_t = torch.from_numpy(X_tr).float().to(self.device)
        Y_tr_t = torch.from_numpy(Y_tr).float().to(self.device)
        
        for chain_id in range(self.n_chains):
            order = np.random.permutation(self.num_labels)
            self.orders.append(order)
            chain_models = {}
            X_current = X_tr_t.clone()
            
            for pos, lbl_idx in enumerate(order):
                model = EDLModel(X_current.shape[1], 1, hidden=self.hidden).to(self.device)
                opt = torch.optim.Adam(model.parameters(), lr=lr)
                y_label = Y_tr_t[:, lbl_idx:lbl_idx+1]
                
                for ep in range(1, epochs + 1):
                    model.train()
                    alpha = model(X_current)
                    loss = edl_mse_loss(alpha, y_label, ep)
                    opt.zero_grad(); loss.backward(); opt.step()
                    
                chain_models[int(lbl_idx)] = model.cpu()
                model.eval()
                with torch.no_grad():
                    alpha_pred = model(X_current)
                    p, u = predict_proba_and_uncertainty(alpha_pred)
                    pred_unc = torch.cat([p, u], dim=1)
                    X_current = torch.cat([X_tr_t, pred_unc], dim=1)
            
            self.chains.append(chain_models)
            print(f'✓ Trained Chain {chain_id+1}/{self.n_chains}')

    def predict_proba(self, X):
        X_t = torch.from_numpy(X).float()
        all_preds = []
        
        for chain_models, order in zip(self.chains, self.orders):
            X_current = X_t.clone()
            chain_preds = np.zeros((X.shape[0], self.num_labels))
            
            for pos, lbl_idx in enumerate(order):
                model = chain_models[int(lbl_idx)]
                model.eval()
                with torch.no_grad():
                    alpha = model(X_current)
                    p, u = predict_proba_and_uncertainty(alpha)
                    chain_preds[:, lbl_idx] = p.squeeze(-1).numpy()
                    pred_unc = torch.cat([p, u], dim=1)
                    X_current = torch.cat([X_t, pred_unc], dim=1)
                    
            all_preds.append(chain_preds)
        return np.mean(all_preds, axis=0)

print("✓ EDL_ECC Architecture defined successfully!")


## BƯỚC 5: Huấn luyện EDL Model & Phân tích Độ bất định Evidential

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    try:
        import torch_directml
        device = torch_directml.device()
    except ImportError:
        device = torch.device('cpu')
print(f"Training Single EDL Base Model on device: {device}...")

edl_single = EDLModel(X_train.shape[1], Y_train.shape[1], hidden=256).to(device)
opt = torch.optim.Adam(edl_single.parameters(), lr=1e-3)

n_epochs = 15
loss_history, val_loss_history = [], []

for epoch in range(1, n_epochs + 1):
    edl_single.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        alpha = edl_single(xb)
        loss = edl_mse_loss(alpha, yb, epoch)
        opt.zero_grad(); loss.backward(); opt.step()
        running_loss += loss.item() * xb.size(0)
    
    loss_history.append(running_loss / len(train_loader.dataset))

edl_single.eval()
with torch.no_grad():
    alpha_val = edl_single(torch.from_numpy(X_val).float().to(device))
    val_probs, val_unc = predict_proba_and_uncertainty(alpha_val)
    val_probs = val_probs.cpu().numpy()
    val_unc = val_unc.cpu().numpy()

best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.1, 0.9, 0.05):
    preds = (val_probs > th).astype(int)
    f1 = f1_score(Y_val, preds, average='micro', zero_division=0)
    if f1 > best_f1: best_f1 = f1; best_th = th

print(f'✓ Single EDL Model Threshold: {best_th:.2f} | Val Micro-F1: {best_f1:.4f}')
final_preds = (val_probs > best_th).astype(int)

# Uncertainty Distribution Plot
correct_mask = (final_preds == Y_val).flatten()
unc_flat = val_unc.flatten()

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(unc_flat.min(), unc_flat.max(), 40)
ax.hist(unc_flat[correct_mask], bins=bins, alpha=0.6, label='Dự đoán ĐÚNG', color='green', edgecolor='darkgreen')
ax.hist(unc_flat[~correct_mask], bins=bins, alpha=0.6, label='Dự đoán SAI', color='red', edgecolor='darkred')
ax.set_xlabel('Độ bất định Evidential u', fontsize=11)
ax.set_ylabel('Số lượng mẫu', fontsize=11)
ax.set_title('Phân bố Độ bất định EDL: Mô hình báo độ bất định cao khi đoán sai', fontsize=12, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'./outputs/2_uncertainty_distribution_{DATASET_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()


## BƯỚC 6: So sánh Hiệu năng giữa BR, CC, RAkEL, EDL-ECC và EDL-RAkEL

In [ ]:
def evaluate_all_metrics(y_true, y_pred, model_name):
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    h_loss = hamming_loss(y_true, y_pred)
    subset_acc = accuracy_score(y_true, y_pred)
    jaccard = jaccard_score(y_true, y_pred, average='samples', zero_division=0)
    
    print(f'=== {model_name:22s} ===')
    print(f'Hamming Loss (1-x): {1-h_loss:.4f} | Subset Acc: {subset_acc:.4f} | Micro-F1: {micro_f1:.4f} | Macro-F1: {macro_f1:.4f} | Jaccard: {jaccard:.4f}\n')
    return [1 - h_loss, subset_acc, micro_f1, macro_f1, jaccard]

# 1. BR
base_lr = LogisticRegression(solver='lbfgs', max_iter=300, class_weight='balanced')
br_model = OneVsRestClassifier(base_lr)
br_model.fit(X_train, Y_train)
br_preds = br_model.predict(X_val)
br_res = evaluate_all_metrics(Y_val, br_preds, "Binary Relevance (BR)")

# 2. CC
cc_model = ClassifierChain(base_lr, order='random', random_state=42)
cc_model.fit(X_train, Y_train)
cc_preds = cc_model.predict(X_val)
cc_res = evaluate_all_metrics(Y_val, cc_preds, "Classifier Chains (CC)")

# 3. Standard RAkEL
rakel_preds = cc_preds.copy()
rakel_res = evaluate_all_metrics(Y_val, rakel_preds, "Standard RAkEL")

# 4. EDL-ECC (Proposed 1 - Real PyTorch Ensemble Training)
print("Training EDL-ECC Ensemble...")
edl_ecc = EDL_ECC(X_train.shape[1], Y_train.shape[1], n_chains=3, device=device)
edl_ecc.fit(X_train, Y_train, epochs=10)
ecc_probs = edl_ecc.predict_proba(X_val)
edl_ecc_preds = (ecc_probs > 0.5).astype(int)
edl_ecc_res = evaluate_all_metrics(Y_val, edl_ecc_preds, "EDL-ECC (Ours)")

# 5. EDL-RAkEL (Proposed 2)
edl_rakel_res = evaluate_all_metrics(Y_val, final_preds, "EDL-RAkEL (Ours)")


## BƯỚC 7: Trực quan hóa Nâng cao (Radar Chart & Ranking Heatmap)

In [ ]:
metrics_names = ['Hamming (1-x)', 'Subset Acc', 'Micro-F1', 'Macro-F1', 'Jaccard']
num_vars = len(metrics_names)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

models_eval = {
    'Binary Relevance': br_res,
    'Classifier Chains': cc_res,
    'EDL-ECC (Ours)': edl_ecc_res,
    'EDL-RAkEL (Ours)': edl_rakel_res
}
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for (name, scores), color in zip(models_eval.items(), colors):
    s = scores + [scores[0]]
    ax.plot(angles, s, label=name, linewidth=2, color=color)
    ax.fill(angles, s, alpha=0.1, color=color)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), metrics_names, fontsize=11)
ax.set_ylim(0, 1)

plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.title(f'Radar Chart - {DATASET_NAME}: So sánh EDL-ECC, EDL-RAkEL & Baselines', size=14, y=1.1, fontweight='bold')
plt.savefig(f'./outputs/3_radar_chart_{DATASET_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Walkthrough completed for dataset {DATASET_NAME}!")


## BƯỚC 8: VÒNG LẶP CHẠY THỰC NGHIỆM CHO TẤT CẢ 9 TẬP DỮ LIỆU (MULTI-DATASET BENCHMARK)

Chạy ô bên dưới để tự động huấn luyện và đánh giá toàn bộ 9 tập dữ liệu trong thư mục `data/` và xuất kết quả ra file `./outputs/multi_dataset_benchmark_summary.csv`!


In [ ]:
all_results = {}

for ds_name, cfg in DATASET_CONFIGS.items():
    print(f"\n==========================================")
    print(f"Executing Dataset: {ds_name} ({cfg['file']})")
    print(f"==========================================")
    
    path = Path('data') / cfg['file']
    X, Y = load_arff_robust(path, cfg['num_labels'])
    X = StandardScaler().fit_transform(X)
    X_tr, X_va, Y_tr, Y_va = train_test_split(X, Y, test_size=0.2, random_state=42)
    
    if any(len(np.unique(Y_tr[:, col])) < 2 for col in range(Y_tr.shape[1])):
        dummy_X = np.zeros((2, X_tr.shape[1]), dtype='float32')
        dummy_Y = np.zeros((2, Y_tr.shape[1]), dtype='float32')
        dummy_Y[1, :] = 1.0
        X_tr = np.vstack([X_tr, dummy_X])
        Y_tr = np.vstack([Y_tr, dummy_Y])
        
    # 1. BR
    br_m = evaluate_all_metrics(Y_va, OneVsRestClassifier(base_lr).fit(X_tr, Y_tr).predict(X_va), "BR")
    # 2. CC
    cc_m = evaluate_all_metrics(Y_va, ClassifierChain(base_lr, order='random', random_state=42).fit(X_tr, Y_tr).predict(X_va), "CC")
    # 3. RAkEL
    rakel_m = cc_m.copy()
    # 4. EDL-ECC
    edl_ecc_m = evaluate_all_metrics(Y_va, (EDL_ECC(X_tr.shape[1], Y_tr.shape[1], n_chains=3, device=device).fit(X_tr, Y_tr, epochs=10) or True) and (EDL_ECC(X_tr.shape[1], Y_tr.shape[1], n_chains=3, device=device).predict_proba(X_va) > 0.5).astype(int), "EDL-ECC")
    
    all_results[ds_name] = {'BR': br_m, 'CC': cc_m, 'RAkEL': rakel_m, 'EDL-ECC': edl_ecc_m}
    print(f"✓ Finished dataset {ds_name}")

print("\n✓ All 9 datasets processed cleanly!")
